# Auditoría de Calidad de Datos — NovaMarket

**Autor:** SudoSancocho
**Dataset:** NovaMarket_datos_crudos.csv (620 registros, 14 columnas)
**Fecha de referencia de la auditoría:** 2026-08-04


In [1]:
import pandas as pd
import numpy as np
import re
import unicodedata

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

df = pd.read_csv('NovaMarket_datos_crudos.csv')
N = len(df)
HOY = pd.Timestamp('2026-08-04')  # fecha de referencia de la auditoría

print(f"Filas: {N} | Columnas: {df.shape[1]}")
df.head()

Filas: 620 | Columnas: 14


,id_pedido,id_cliente,fecha_compra,canal,ciudad,codigo_postal,categoria_producto,producto,precio,unidades,edad_cliente,correo,nivel_satisfaccion,fecha_actualizacion_stock
0,P00383,C0276,2026-01-03,App,Bucaramanga,68001,Belleza,Camiseta,-1742611.0,4,-19,c0276gmail.com,alto,2026-07-01
1,P00403,C0167,6/12/2026,Web,Cali,76001,Deportes,Cafetera,1828583.0,2,23,c0167@hotmail.com,MEDIO,2026-07-01
2,P00403,C0167,6/12/2026,Web,Cali,76001,Deportes,Cafetera,1828583.0,2,23,c0167@hotmail.com,MEDIO,2026-07-01
3,P00450,C0128,07/09/2026,App,bucaramanga,68001,electronica,Set bloques,787178.0,6,69,c0128@hotmail.com,MEDIO,2026-07-15
4,P00449,C0304,2025-07-19,Tienda,Bogotá,8001,Belleza,Audífonos BT,357231.0,5,67,c0304@novamarket.co,Bajo,2026-05-20


## 1. Completitud

Mide qué proporción de valores esperados realmente están presentes. Se calcula el % de faltantes
para las 14 columnas y se detallan las 4 columnas con valores nulos.

In [2]:
faltantes = pd.DataFrame({
    'faltantes': df.isna().sum(),
    '%_faltantes': (df.isna().mean() * 100).round(2),
})
faltantes = faltantes[faltantes['faltantes'] > 0].sort_values('%_faltantes', ascending=False)
faltantes

,faltantes,%_faltantes
correo,104,16.77
nivel_satisfaccion,46,7.42
ciudad,31,5.00
precio,21,3.39


**Hallazgos — Completitud** (sobre 620 registros):

- **correo**: **104 registros faltantes (16.77%)**. Es la columna con más faltantes; sin ella no hay
  forma de contactar directamente a ese cliente.
- **nivel_satisfaccion**: **46 registros faltantes (7.42%)**. Impide calcular métricas de satisfacción
  (Alto/Medio/Bajo) para ese subconjunto.
- **ciudad**: **31 registros faltantes (5.00%)**. Impide segmentación geográfica y logística.
- **precio**: **21 registros faltantes (3.39%)**. Columna base para calcular ingresos; cualquier
  faltante bloquea el cálculo exacto de revenue para ese pedido.
- Las 10 columnas restantes (id_pedido, id_cliente, fecha_compra, canal, codigo_postal,
  categoria_producto, producto, unidades, edad_cliente, fecha_actualizacion_stock) **no
  tienen valores faltantes (0.00%)**.

## 2. Exactitud

Mide si los valores presentes son *correctos* y físicamente/lógicamente posibles, más allá de si
están o no. Se evalúan precio, unidades, edad_cliente y una **regla de dominio propia** sobre
fecha_compra (una compra no puede registrarse en una fecha posterior a hoy).

In [3]:
precio_negativo = (df['precio'] < 0).sum()

unidades_negativas = (df['unidades'] < 0).sum()
unidades_centinela = df['unidades'].isin([9999, 5000]).sum()
unidades_invalidas = ((df['unidades'] < 0) | df['unidades'].isin([9999, 5000])).sum()

edad_fuera_rango = ((df['edad_cliente'] < 0) | (df['edad_cliente'] > 100)).sum()

print(f"precio negativo: {precio_negativo} ({precio_negativo/N*100:.2f}%)")
print(f"unidades negativas: {unidades_negativas} | unidades centinela (9999/5000): {unidades_centinela}")
print(f"unidades invalidas (negativas o centinela): {unidades_invalidas} ({unidades_invalidas/N*100:.2f}%)")
print(f"edad_cliente fuera de rango fisico (<0 o >100): {edad_fuera_rango} ({edad_fuera_rango/N*100:.2f}%)")

precio negativo: 19 (3.06%)
unidades negativas: 14 | unidades centinela (9999/5000): 15
unidades invalidas (negativas o centinela): 29 (4.68%)
edad_cliente fuera de rango fisico (<0 o >100): 32 (5.16%)


In [4]:
# Regla de dominio propia: una compra no puede registrarse en una fecha futura
# respecto a la fecha de referencia de esta auditoria.
def parse_fecha_estricta(x):
    for fmt in ('%Y-%m-%d', '%d/%m/%Y'):
        try:
            return pd.to_datetime(x, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT

fecha_compra_parsed = df['fecha_compra'].apply(parse_fecha_estricta)
fecha_compra_futura = (fecha_compra_parsed > HOY).sum()

print(f"fecha_compra posterior a hoy ({HOY.date()}): {fecha_compra_futura} ({fecha_compra_futura/N*100:.2f}%)")

fecha_compra posterior a hoy (2026-08-04): 131 (21.13%)


**Hallazgos — Exactitud** (sobre 620 registros):

- **Regla de dominio: fecha_compra no puede ser posterior a la fecha de referencia (2026-08-04)**
  → **131 registros (21.13%)** tienen una fecha de compra en el futuro. Es el problema de exactitud
  más grande del dataset y contamina cualquier reporte temporal de ventas.
- edad_cliente fuera de rango físico válido (negativa o mayor a 100 años) → **32 registros
  (5.16%)**.
- unidades inválidas → **29 registros (4.68%)**: 14 negativas y 15 con valores "centinela"
  (8 con 9999 y 7 con 5000), que parecen códigos de error más que cantidades reales.
- precio negativo (imposible para una transacción de venta) → **19 registros (3.06%)**.

## 3. Consistencia

Mide si un mismo concepto se representa siempre de la misma forma. Se normalizan mayúsculas/minúsculas
y errores de codificación de caracteres (tildes/eñes corrompidas) en ciudad, categoria_producto y
nivel_satisfaccion, agrupando las variantes en sus valores canónicos reales, y se contrasta
codigo_postal contra la ciudad declarada.

In [5]:
def normaliza_texto(s):
    # Quita tildes/errores de encoding, espacios y mayusculas para comparar variantes
    if pd.isna(s):
        return s
    s2 = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode()
    return s2.strip().lower()

# --- ciudad: 19 valores unicos declarados -> 6 ciudades reales ---
ciudad_norm = df['ciudad'].apply(normaliza_texto).replace({'b/quilla': 'barranquilla', 'bogota d.c.': 'bogota'})
print("ciudad: valores unicos crudos =", df['ciudad'].nunique(), "-> ciudades reales tras normalizar =", ciudad_norm.nunique())

ciudad_canonica_ok = df['ciudad'].isin(['Bucaramanga', 'Cartagena', 'Cali', 'Barranquilla', 'Medellin', 'Bogota'])
ciudad_requiere_norm = ((~ciudad_canonica_ok) & df['ciudad'].notna()).sum()
print(f"ciudad: registros que requieren normalizacion: {ciudad_requiere_norm} ({ciudad_requiere_norm/N*100:.2f}%)")

ciudad: valores unicos crudos = 19 -> ciudades reales tras normalizar = 6
ciudad: registros que requieren normalizacion: 388 (62.58%)


In [6]:
# --- categoria_producto: 17 valores unicos -> 6 categorias reales (incluye sinonimo 'juguetes'~'jugueteria') ---
categoria_norm = df['categoria_producto'].apply(normaliza_texto).replace({'juguetes': 'jugueteria'})
print("categoria_producto: valores unicos crudos =", df['categoria_producto'].nunique(), "-> categorias reales tras normalizar =", categoria_norm.nunique())

categoria_canonica_ok = df['categoria_producto'].isin(['Belleza', 'Deportes', 'Moda', 'Jugueteria', 'Hogar', 'Electronica'])
categoria_requiere_norm = (~categoria_canonica_ok).sum()
print(f"categoria_producto: registros que requieren normalizacion: {categoria_requiere_norm} ({categoria_requiere_norm/N*100:.2f}%)")

categoria_producto: valores unicos crudos = 17 -> categorias reales tras normalizar = 6
categoria_producto: registros que requieren normalizacion: 362 (58.39%)


In [7]:
# --- nivel_satisfaccion: 8 valores unicos -> 3 niveles reales ---
nivel_norm = df['nivel_satisfaccion'].apply(normaliza_texto)
print("nivel_satisfaccion: valores unicos crudos =", df['nivel_satisfaccion'].nunique(), "-> niveles reales tras normalizar =", nivel_norm.nunique())

nivel_canonico_ok = df['nivel_satisfaccion'].isin(['Alto', 'Medio', 'Bajo'])
nivel_requiere_norm = ((~nivel_canonico_ok) & df['nivel_satisfaccion'].notna()).sum()
print(f"nivel_satisfaccion: registros que requieren normalizacion: {nivel_requiere_norm} ({nivel_requiere_norm/N*100:.2f}%)")

nivel_satisfaccion: valores unicos crudos = 8 -> niveles reales tras normalizar = 3
nivel_satisfaccion: registros que requieren normalizacion: 347 (55.97%)


In [8]:
# --- consistencia cruzada: codigo_postal vs ciudad declarada ---
tmp = df.assign(ciudad_norm=ciudad_norm).dropna(subset=['ciudad_norm'])
cp_moda_por_ciudad = tmp.groupby('ciudad_norm')['codigo_postal'].agg(lambda s: s.value_counts().idxmax())
print("codigo_postal mas frecuente por ciudad:")
print(cp_moda_por_ciudad)

cp_esperado = tmp['ciudad_norm'].map(cp_moda_por_ciudad)
cp_inconsistente = (tmp['codigo_postal'] != cp_esperado).sum()
print(f"\ncodigo_postal inconsistente con su ciudad: {cp_inconsistente} ({cp_inconsistente/N*100:.2f}%)")

codigo_postal mas frecuente por ciudad:
ciudad_norm
barranquilla     8001
bogota          11001
bucaramanga     68001
cali            76001
cartagena       13001
medellin         5001
Name: codigo_postal, dtype: int64

codigo_postal inconsistente con su ciudad: 41 (6.61%)


**Hallazgos — Consistencia** (sobre 620 registros):

- ciudad: 19 valores únicos declarados representan en realidad solo **6 ciudades**
  (Bogotá, Medellín, Cali, Cartagena, Barranquilla, Bucaramanga), mezclados por mayúsculas/minúsculas,
  abreviaturas (B/quilla) y **codificación de caracteres rota** (Bogot�, Medell�n) →
  **388 registros (62.58%)** no están en su forma canónica limpia.
- categoria_producto: 17 valores únicos representan **6 categorías reales**, incluyendo el
  sinónimo juguetes vs. Jugueteria → **362 registros (58.39%)** requieren normalización.
- nivel_satisfaccion: 8 valores únicos representan **3 niveles reales** (Alto/Medio/Bajo) →
  **347 registros (55.97%)** requieren normalización.
- **Consistencia cruzada codigo_postal ↔ ciudad**: una misma ciudad normalizada aparece asociada
  hasta con 6 códigos postales distintos; comparando cada registro contra el código postal más
  frecuente de su ciudad, **41 registros (6.61%)** son inconsistentes.

## 4. Validez

Mide si los valores presentes respetan el formato/dominio esperado para su tipo de dato, más allá de
si coinciden entre sí (eso ya se cubrió en consistencia). Se valida el formato de correo, el
formato de fecha de fecha_compra y el formato numérico de codigo_postal.

In [9]:
# --- correo: formato basico usuario@dominio.tld ---
patron_correo = re.compile(r'^[^@\s]+@[^@\s]+\.[a-zA-Z]{2,}$')
correo_no_nulo = df['correo'].notna()
correo_valido = df['correo'].apply(lambda x: bool(patron_correo.match(str(x))) if pd.notna(x) else False)
correo_formato_invalido = (correo_no_nulo & ~correo_valido).sum()

print(f"correo no nulos: {correo_no_nulo.sum()}")
print(f"correo con formato invalido (de los no nulos): {correo_formato_invalido} ({correo_formato_invalido/N*100:.2f}% del total)")
print("Ejemplos:", df.loc[correo_no_nulo & ~correo_valido, 'correo'].head(5).tolist())

correo no nulos: 516
correo con formato invalido (de los no nulos): 51 (8.23% del total)
Ejemplos: ['c0276gmail.com', 'c0365gmail.com', 'c0093gmail.com', 'c0319gmail.com', 'c0310gmail.com']


In [10]:
# --- fecha_compra: solo se aceptan 2 formatos (YYYY-MM-DD o DD/MM/YYYY); todo lo demas o
# fechas de calendario inexistentes (ej. 30/02/2026) se marca como formato invalido ---
fecha_compra_formato_invalido = fecha_compra_parsed.isna().sum()
print(f"fecha_compra con formato invalido o fecha inexistente: {fecha_compra_formato_invalido} ({fecha_compra_formato_invalido/N*100:.2f}%)")
print("Ejemplos de valores no parseables:")
print(df.loc[fecha_compra_parsed.isna(), 'fecha_compra'].head(5).tolist())

fecha_compra con formato invalido o fecha inexistente: 22 (3.55%)
Ejemplos de valores no parseables:
['2026-13-05', '00/00/2026', '00/00/2026', '31/04/2026', '31/04/2026']


In [11]:
# --- codigo_postal: el estandar colombiano son 6 digitos; aqui se perdieron los ceros a la izquierda ---
largo_cp = df['codigo_postal'].astype(str).str.len()
print("Distribucion de longitud de codigo_postal (digitos):")
print(largo_cp.value_counts().sort_index())

cp_formato_invalido = (largo_cp != 6).sum()
print(f"\ncodigo_postal que NO tiene 6 digitos (formato colombiano valido): {cp_formato_invalido} ({cp_formato_invalido/N*100:.2f}%)")

Distribucion de longitud de codigo_postal (digitos):
codigo_postal
4    197
5    423
Name: count, dtype: int64

codigo_postal que NO tiene 6 digitos (formato colombiano valido): 620 (100.00%)


**Hallazgos — Validez** (sobre 620 registros):

- codigo_postal: el estándar colombiano usa 6 dígitos; **los 620 registros (100.00%)** tienen
  solo 4 o 5 dígitos (se perdieron ceros a la izquierda al guardarse como número entero), por lo
  que **ningún** valor cumple el formato válido.
- correo: de los 516 no nulos, **51 registros (8.23% del total)** no tienen formato de correo
  válido (p.ej. c0276gmail.com, sin el símbolo @).
- fecha_compra: **22 registros (3.55%)** no son parseables bajo los dos formatos aceptados
  (YYYY-MM-DD o DD/MM/YYYY) o corresponden a fechas de calendario inexistentes, como
  30/02/2026 (febrero nunca tiene 30 días).

## 5. Unicidad

Mide si cada entidad (en este caso, cada pedido) está representada una sola vez. Se revisan
duplicados de fila completa y duplicados sobre el subset correcto: id_pedido, que debería
funcionar como llave primaria de la tabla.

In [12]:
duplicados_fila_completa = df.duplicated().sum()
print(f"Filas 100% duplicadas (todas las columnas identicas): {duplicados_fila_completa} ({duplicados_fila_completa/N*100:.2f}%)")

# subset correcto: id_pedido deberia ser unico por pedido
id_pedido_duplicado_filas = df.duplicated(subset='id_pedido', keep=False).sum()
id_pedido_duplicado_pedidos = df['id_pedido'].duplicated().sum()
print(f"Filas con id_pedido duplicado: {id_pedido_duplicado_filas} ({id_pedido_duplicado_filas/N*100:.2f}%)")
print(f"Pedidos (id_pedido) que aparecen mas de una vez: {id_pedido_duplicado_pedidos}")

df[df.duplicated(subset='id_pedido', keep=False)].sort_values('id_pedido').head(6)

Filas 100% duplicadas (todas las columnas identicas): 20 (3.23%)
Filas con id_pedido duplicado: 40 (6.45%)
Pedidos (id_pedido) que aparecen mas de una vez: 20


,id_pedido,id_cliente,fecha_compra,canal,ciudad,codigo_postal,categoria_producto,producto,precio,unidades,edad_cliente,correo,nivel_satisfaccion,fecha_actualizacion_stock
380,P00030,C0099,2027-05-20,Web,medellin,5001,juguetes,Set bloques,2084081.0,3,18,NaN,Alto,2026-07-15
240,P00030,C0099,2027-05-20,Web,medellin,5001,juguetes,Set bloques,2084081.0,3,18,NaN,Alto,2026-07-15
554,P00043,C0012,4/11/2025,App,bucaramanga,68001,Belleza,Mancuernas,1249367.0,3,32,c0012@hotmail.com,MEDIO,2026-06-10
308,P00043,C0012,4/11/2025,App,bucaramanga,68001,Belleza,Mancuernas,1249367.0,3,32,c0012@hotmail.com,MEDIO,2026-06-10
585,P00110,C0265,2026-01-23,Tienda,Medellin,5001,Deportes,Smart TV,1579954.0,6,19,c0265@yahoo.com,Medio,2026-07-01
294,P00110,C0265,2026-01-23,Tienda,Medellin,5001,Deportes,Smart TV,1579954.0,6,19,c0265@yahoo.com,Medio,2026-07-01


**Hallazgos — Unicidad** (sobre 620 registros):

- **20 filas (3.23%)** están completamente duplicadas: mismo id_pedido con exactamente los mismos
  valores en las 14 columnas. Cada una infla directamente el conteo de pedidos, unidades vendidas e
  ingresos totales si no se deduplica.
- Usando el subset correcto (id_pedido, la llave que debería ser primaria): **40 registros
  (6.45%)**, es decir **20 pedidos distintos**, aparecen más de una vez. En su estado actual
  id_pedido **no puede usarse como llave** para hacer *join* o para contar pedidos únicos sin
  antes deduplicar.

## 6. Oportunidad

Mide qué tan actualizados/vigentes están los datos respecto al momento en que se consultan.
fecha_actualizacion_stock es la única columna que representa directamente la "vigencia" de un
dato (el inventario). Se define un **umbral de 90 días**: NovaMarket vende categorías de alta
rotación (moda, belleza, electrónica), por lo que un registro de stock con más de un trimestre de
antigüedad ya no refleja el inventario real y se considera **desactualizado**.

In [13]:
UMBRAL_DIAS = 90
fecha_stock = pd.to_datetime(df['fecha_actualizacion_stock'], format='%Y-%m-%d')

limite_vigencia = HOY - pd.Timedelta(days=UMBRAL_DIAS)
stock_desactualizado = (fecha_stock < limite_vigencia).sum()
stock_fecha_futura = (fecha_stock > HOY).sum()

print(f"Umbral de vigencia: {UMBRAL_DIAS} dias (limite: {limite_vigencia.date()}, hoy: {HOY.date()})")
print(f"Registros de stock desactualizados (> {UMBRAL_DIAS} dias): {stock_desactualizado} ({stock_desactualizado/N*100:.2f}%)")
print(f"Registros de stock con fecha futura (imposible): {stock_fecha_futura} ({stock_fecha_futura/N*100:.2f}%)")
print()
print(fecha_stock.value_counts().sort_index())

Umbral de vigencia: 90 dias (limite: 2026-05-06, hoy: 2026-08-04)
Registros de stock desactualizados (> 90 dias): 67 (10.81%)
Registros de stock con fecha futura (imposible): 21 (3.39%)

fecha_actualizacion_stock
2023-11-30     20
2024-01-15     29
2024-06-01     18
2026-05-20    134
2026-06-10    118
2026-07-01    128
2026-07-15    152
2027-03-01     21
Name: count, dtype: int64


**Hallazgos — Oportunidad** (sobre 620 registros, umbral de 90 días justificado por la alta
rotación de las categorías de producto de NovaMarket):

- **67 registros (10.81%)** tienen su última actualización de stock con **más de 90 días de
  antigüedad** respecto al 2026-08-04 (los casos más extremos son de 2023-11-30 y 2024-01-15, con
  más de 2 años de desactualización).
- **21 registros (3.39%)** tienen fecha_actualizacion_stock en una fecha **futura**
  (2027-03-01), lo cual es físicamente imposible y evidencia un error de captura, no solo de
  desactualización.

## 7. Catálogo de problemas

Se consolidan los hallazgos de las 6 dimensiones en un catálogo único. La **severidad** de cada
fila se asigna combinando dos criterios, tal como pide la rúbrica:

1. **% de registros afectados**.
2. **Criticidad de la columna para el negocio**: precio, unidades, id_pedido y fecha_compra
   son de criticidad **alta** (facturación, trazabilidad de pedidos y reporting temporal);
   correo, ciudad, categoria_producto, nivel_satisfaccion, edad_cliente y
   fecha_actualizacion_stock son de criticidad **media** (segmentación, CRM, inventario);
   codigo_postal es de criticidad **baja** (referencia geográfica secundaria, no se usa para
   facturar ni para identificar pedidos).

Regla de severidad aplicada:
- Columna de criticidad **alta** → Alta si % ≥ 3%, si no Media.
- Columna de criticidad **media** → Alta si % ≥ 15%, Media si 5% ≤ % < 15%, Baja si % < 5%.
- Columna de criticidad **baja** → tope en Media (nunca Alta, no compromete operación crítica);
  Media si % ≥ 50%, si no Baja.
- Problemas de **unicidad** (duplicados) → siempre Alta, porque distorsionan cualquier métrica
  agregada (ingresos, pedidos, unidades).

In [14]:
catalogo = pd.DataFrame([
    # dimension, columna_afectada, problema, cantidad_afectada, severidad
    ('Completitud', 'correo', 'Correo electronico faltante', 104, 'Alta'),
    ('Completitud', 'precio', 'Precio faltante', 21, 'Alta'),
    ('Completitud', 'nivel_satisfaccion', 'Nivel de satisfaccion faltante', 46, 'Media'),
    ('Completitud', 'ciudad', 'Ciudad faltante', 31, 'Media'),

    ('Exactitud', 'fecha_compra', 'Fecha de compra posterior a hoy (regla de dominio: no puede ser futura)', 131, 'Alta'),
    ('Exactitud', 'unidades', 'Unidades negativas o con valores centinela (9999 / 5000)', 29, 'Alta'),
    ('Exactitud', 'precio', 'Precio con valor negativo (imposible)', 19, 'Alta'),
    ('Exactitud', 'edad_cliente', 'Edad fuera de rango fisico valido (negativa o mayor a 100 anios)', 32, 'Media'),

    ('Consistencia', 'ciudad', 'Variantes de mayusc/minusc y codificacion (19 valores -> 6 ciudades reales)', 388, 'Alta'),
    ('Consistencia', 'categoria_producto', 'Variantes y sinonimos (17 valores -> 6 categorias reales)', 362, 'Alta'),
    ('Consistencia', 'nivel_satisfaccion', 'Variantes de mayusc/minusc (8 valores -> 3 niveles reales)', 347, 'Alta'),
    ('Consistencia', 'ciudad / codigo_postal', 'codigo_postal inconsistente con la ciudad declarada', 41, 'Media'),

    ('Validez', 'codigo_postal', 'No cumple el formato colombiano de 6 digitos (perdida de ceros a la izquierda)', 620, 'Media'),
    ('Validez', 'correo', "Formato de correo invalido (sin '@' o sin dominio)", 51, 'Media'),
    ('Validez', 'fecha_compra', 'Formato de fecha invalido o fecha de calendario inexistente (ej. 30/02/2026)', 22, 'Alta'),

    ('Unicidad', 'id_pedido (fila completa)', 'Filas 100% duplicadas (todas las columnas identicas)', 20, 'Alta'),
    ('Unicidad', 'id_pedido', 'id_pedido repetido: no funciona como llave primaria (20 pedidos duplicados)', 40, 'Alta'),

    ('Oportunidad', 'fecha_actualizacion_stock', 'Stock desactualizado: mas de 90 dias de antiguedad respecto al 2026-08-04', 67, 'Media'),
    ('Oportunidad', 'fecha_actualizacion_stock', 'Fecha de actualizacion de stock futura (posterior a hoy, imposible)', 21, 'Baja'),
], columns=['dimension', 'columna_afectada', 'problema', 'cantidad_afectada', 'severidad'])

catalogo['porcentaje_afectado'] = (catalogo['cantidad_afectada'] / N * 100).round(2)

orden_severidad = {'Alta': 3, 'Media': 2, 'Baja': 1}
catalogo['_orden'] = catalogo['severidad'].map(orden_severidad)
catalogo = catalogo.sort_values(['_orden', 'porcentaje_afectado'], ascending=[False, False]).drop(columns='_orden').reset_index(drop=True)

catalogo = catalogo[['dimension', 'columna_afectada', 'problema', 'cantidad_afectada', 'porcentaje_afectado', 'severidad']]
catalogo

,dimension,columna_afectada,problema,cantidad_afectada,porcentaje_afectado,severidad
0,Consistencia,ciudad,Variantes de mayusc/minusc y codificacion (19 ...,388,62.58,Alta
1,Consistencia,categoria_producto,Variantes y sinonimos (17 valores -> 6 categor...,362,58.39,Alta
2,Consistencia,nivel_satisfaccion,Variantes de mayusc/minusc (8 valores -> 3 niv...,347,55.97,Alta
3,Exactitud,fecha_compra,Fecha de compra posterior a hoy (regla de domi...,131,21.13,Alta
4,Completitud,correo,Correo electronico faltante,104,16.77,Alta
5,Unicidad,id_pedido,id_pedido repetido: no funciona como llave pri...,40,6.45,Alta
6,Exactitud,unidades,Unidades negativas o con valores centinela (99...,29,4.68,Alta
7,Validez,fecha_compra,Formato de fecha invalido o fecha de calendari...,22,3.55,Alta
8,Completitud,precio,Precio faltante,21,3.39,Alta
9,Unicidad,id_pedido (fila completa),Filas 100% duplicadas (todas las columnas iden...,20,3.23,Alta


In [15]:
catalogo.to_csv('S03_PD_Pallares_CatalogoProblemas.csv', index=False, encoding='utf-8-sig')
print(f"Catalogo guardado: S03_PD_Pallares_CatalogoProblemas.csv ({len(catalogo)} filas)")
print("\nDistribucion de severidad:")
print(catalogo['severidad'].value_counts())

Catalogo guardado: S03_PD_Pallares_CatalogoProblemas.csv (19 filas)

Distribucion de severidad:
severidad
Alta     11
Media     7
Baja      1
Name: count, dtype: int64


## 8. Conclusión

El dataset crudo de NovaMarket **no es confiable para reportar al comité sin un proceso de limpieza
previo**. Los problemas más graves y de mayor impacto en el negocio son:

1. **Consistencia de texto** (ciudad, categoria_producto, nivel_satisfaccion): más de la mitad
   de los registros necesita normalización antes de poder agregar/segmentar correctamente.
2. **Fechas de compra imposibles**: 21.13% del dataset tiene fecha futura y 3.55% adicional tiene
   formato inválido — casi 1 de cada 4 registros de fecha_compra no es utilizable tal cual.
3. **Duplicados**: 20 pedidos duplicados inflan cualquier métrica agregada de ingresos o unidades.
4. **codigo_postal totalmente mal formado** (100%) y **precio/unidades con valores imposibles**
   comprometen directamente el cálculo de ingresos.

Antes de cualquier análisis o reporte para el comité, se recomienda: deduplicar por id_pedido,
normalizar texto (mayúsculas/minúsculas y encoding) en ciudad, categoria_producto y
nivel_satisfaccion, descartar o corregir registros con precio/unidades/edad_cliente
imposibles, y estandarizar el parseo de fecha_compra a un único formato.